In [ ]:
#定义奖励模型
import torch
from typing import Optional
from torch import nn
import numpy as np
import os
from sympy import false
from transformers import AutoModelForCausalLM, AutoTokenizer

class RewardHead(nn.Module):
    """
    RewardHead类给GPT2实现了一个“头”，为每个输出的token返回一个标量值。
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(
            self.reward.weight,
            std=(1.0 / np.sqrt(self.hidden_size + 1))
        )
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.reward(output)


class GPT2RewardModel(nn.Module):
    """
    GPT2模型加上一个“奖励头”
    """

    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        # 添加奖励头
        self.reward_head = RewardHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:
        # GPT2的输出
        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )

        # 获取最后一层隐藏层
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # 对隐藏层给出奖励
        rewards = self.reward_head(last_hidden_state).squeeze(-1)
        # 归一化
        return torch.sigmoid(rewards)

#加载模型
model_path = "C:\\Users\lihaodong\.cache\modelscope\hub\models\Fengshenbang\Wenzhong-GPT2-110M-chinese-v2"
model = GPT2RewardModel(model_path)
if os.path.exists('models/best_reward_model.pt'):
    model.load_state_dict(torch.load('models/best_reward_model.pt'))
    print('reward_model.pt loaded')

